In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt 
import seaborn as sns
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

        
import warnings
warnings.filterwarnings('ignore')
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
x_train = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/train.csv", header= None)
y_train = pd.read_csv("/kaggle/input/data-science-london-scikit-learn/trainLabels.csv", header = None, names = ["label"])

In [ ]:
x_train.head()

# **notice that our data is numercal**

In [ ]:
test = pd.read_csv('../input/data-science-london-scikit-learn/test.csv', header=None)

In [ ]:
print('Train data:', x_train.shape)
print('Test data:', test.shape)
print('Train label', y_train.shape)

**the testing data is 9 times smaller than the training data**

In [ ]:
x_train.head()

In [ ]:
x_train.info()

**no missing and all of our data is floated no categories**

In [ ]:
y_train.head()

In [ ]:
plt.pie(y_train.value_counts())
plt.show()

**we made sure that the train data is not skewed**

In [ ]:
x_train.describe().T

our data seems scalled 

In [ ]:
fig, ax = plt.subplots(figsize=[15,5])  
plt.boxplot(x_train)
plt.show()

**making sure we have no outliers**

**Too many columns to look for the outlier in the data so I'll use a function to count them in each column**

In [ ]:
def count_outliers(df):
    outlier_count=[]
    for i in df.columns:
        q1= df[i].quantile(.25)
        q3 = df[i].quantile(.75)
        IQR = q3-q1
        lower= q1-1.5*IQR
        upper = q3+ 1.5*IQR
        outliers = ((df[i]>upper)|(df[i]<lower)).sum()
        outlier_count.append(outliers)
    return np.array(outlier_count)

In [ ]:
print(count_outliers(x_train))

there is some outliers but we want to create a generaize model to predict 9000 samples so I'll keep it 

In [ ]:
from scipy.stats import norm
for i in x_train.columns:
    sns.distplot(x_train[i], fit=norm);
    fig = plt.figure()

it looks like our data is normally disributed 

In [ ]:
def draw_correlations(df):
    correlations = df.corr()
    correlations = correlations[correlations > 0.7]
    sns.set(rc={'figure.figsize': (6, 6)})
    sns.heatmap(correlations)
    
draw_correlations(x_train)

the correlation between the columns is lower than .6 so I won't drop any columns 

In [ ]:
#x_train = x_train.drop([4,12], axis=1)
#draw_correlations(x_train)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
#x_train = pd.DataFrame(scaler.fit_transform(x_train))
#x_train.describe().T

I choose to not to scale the data since I think its uneeded also i tried to scale it worked bad with pca and guession mixture that i wanna use later 

In [ ]:
test.info()

now we are looking for the test data to make sure it hase the same distribution as the train 

In [ ]:
x_train.describe().T

In [ ]:
test.describe().T

****as you can see the test data set has similar std and range to the train dataset which actully is excepected from generative data 

In [ ]:
fig, ax = plt.subplots(figsize=[15,5])  
plt.boxplot(test)
plt.show()

In [ ]:
test.info()

In [ ]:
def draw_correlations(df):
    correlations = df.corr()
    correlations = correlations[correlations > 0.7]
    sns.set(rc={'figure.figsize': (6, 6)})
    sns.heatmap(correlations)
    
draw_correlations(test)

In [ ]:
#test = test.drop([4,12], axis=1)
#draw_correlations(test)

we won't drop anything as we see the distribution and the correlation of the test is so simlar to the train

# our data is well scalled and preprocessed it does not need any change its perfect so I'll just go straight to modeling 
 I have decided to try 4 models 
*  1- logistic regrission 
*  2- Random Forest 
*  3- K-nearest neighbeor classifier 
*  4- support vector machine -linear and rbf kernal-

In [ ]:
from sklearn.model_selection import train_test_split
#train_label = np.ravel(train_label)
xtrain, x_val, ytrain, y_val = train_test_split(x_train, y_train, random_state=42, test_size=0.2)
print(xtrain.shape)
print(x_val.shape)

we start with splitting train set to train and validation sets 

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

the base model will be LR 

In [ ]:
LR_model = LogisticRegression()
LR_model.fit(xtrain, ytrain)
y_pred= LR_model.predict(x_val)
lr_acc  = accuracy_score(y_val,y_pred)
print('validation accuracy :{}'.format(lr_acc))

the accuarcy isn't bad but we want better so we will try with the K-nearest neighbour 

In [ ]:
num_neighs = np.arange(1,26)
train_accuracy = []
val_accuracy = []
best_accuracy_knn = 0.0
for i, n in enumerate(num_neighs):
    knn = KNeighborsClassifier(n_neighbors=n)
    knn.fit(xtrain, ytrain)
    train_accuracy.append(knn.score(xtrain, ytrain))
    val_accuracy.append(knn.score(x_val, y_val))
    print("kNN (k={}): train accuracy={:.5f}, validation accuracy={:.5f}"
          .format(n, train_accuracy[i], val_accuracy[i]))
    if val_accuracy[i] > best_accuracy_knn:
        best_accuracy_knn = val_accuracy[i]
        best_knn = knn
        best_num_neighs = n

plt.plot(num_neighs, 1-np.array(train_accuracy), label='Train Error')
plt.plot(num_neighs, 1-np.array(val_accuracy), label='Validation Error')
plt.ylabel('Error')
plt.xlabel('Number of neighbors')
plt.legend()
plt.show()

print('Best validation Error (k={}): {:.5f}'.format(best_num_neighs, best_accuracy_knn))

its accuarcy was much better
for RF i choose to do grid search for its parameters since it has several hyperparms 

In [ ]:
estimators = [25, 50, 75, 100]  
max_depths = [20, 30, 40, 50]
splits = [5, 10, 15]
param = {'n_estimators':estimators, 
         'max_depth':max_depths, 
         'min_samples_split':splits}
train_accuracy = []
val_accuracy = []

rf = RandomForestClassifier(random_state=100)
RandomForestCV = GridSearchCV(estimator=rf, param_grid=param, cv=3)
RandomForestCV.fit(x_train, y_train)
train_accuracy = RandomForestCV.score(x_train, y_train)
best_accuracy_rf = RandomForestCV.score(x_val, y_val)

print(RandomForestCV.best_params_)
print("RF: train accuracy={:.5f}, validation accuracy={:.5f}"
      .format(train_accuracy, best_accuracy_rf))


RF as excepected will over fit on the data and with the gridsearch it gave me better accuarcy

In [ ]:
svm = SVC(kernel='linear', random_state=100, C=1)
svm.fit(x_train, y_train)
train_accuracy = svm.score(x_train, y_train)
best_accuracy_svml = svm.score(x_val, y_val)
print("SVM (linear): train accuracy={:.5f}, validation accuracy={:.5f}"
      .format(train_accuracy, best_accuracy_svml))

SVM linear is not that powerful since we has too many feature  but its still better than LR

In [ ]:
svm = SVC(kernel='rbf', gamma='auto', random_state=100, C=1)
svm.fit(x_train, y_train)
# Train accuracy
train_accuracy = svm.score(x_train, y_train)
# Validation accuracy
best_accuracy_svmr = svm.score(x_val, y_val)
print("SVM (rbf): train accuracy={:.5f}, validation accuracy={:.5f}"
      .format(train_accuracy, best_accuracy_svmr))

as excepected the SVM rbf kernel will give the best accuarcy 

In [ ]:
print('LR: {:.5f}'.format(lr_acc))
print('KNN: {:.5f}'.format(best_accuracy_knn))
print('RF: {:.5f}'.format(best_accuracy_rf))
print('SVM (linear): {:.5f}'.format(best_accuracy_svml))
print('SVM (rbf): {:.5f}'.format(best_accuracy_svmr))

in summary we should go with either svm rbf or RF 

In [ ]:
svm = SVC(kernel='rbf', random_state=100, C=1)
svm.fit(x_train, y_train)
ypred = svm.predict(test)
submission = {'Id': np.arange(1, ypred.shape[0]+1),
             'Solution': ypred}
submission = pd.DataFrame(submission)
submission.to_csv('submission_svm_rbf.csv', index=False)

using svm gave me 90% accuarcy even though it gave me very high validation and training score and since i wanted to try GaussianMixture and PCA 

In [ ]:
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

pca = PCA(0.95,whiten=True)
train_pc = pca.fit_transform(x_train)
test_pc = pca.transform(test)
print(train_pc.shape)
print(test_pc.shape)

we first start with reducing the dimention of the data taking only the important features aka 95% 

In [ ]:
n_components = np.arange(1,11)
models = [GaussianMixture(n, covariance_type='full', random_state=100) 
          for n in n_components]
aics = [model.fit(train_pc).aic(train_pc) for model in models]
bics = [model.fit(train_pc).bic(train_pc) for model in models]

plt.plot(n_components, aics, label='AIC')
plt.plot(n_components, bics, label='BIC')
plt.xlabel('Number of Gaussians')
plt.legend()
plt.show()

now we try to evaluate the GaussianMixture with different numbers of aic and bics to determenie the number of components we should use for our mixture 
and its obviously 4 

In [ ]:
best_gmm = GaussianMixture(n_components=4, covariance_type='full').fit(train_pc)

train_gmm = best_gmm.predict_proba(train_pc)
test_gmm = best_gmm.predict_proba(test_pc)
print(train_gmm.shape)
print(test_gmm.shape)

xtrain_gmm, xval_gmm, ytrain_gmm, yval_gmm = train_test_split(train_gmm, y_train, random_state=42, test_size=0.2)

In [ ]:
pd.DataFrame(xtrain_gmm)

now we predict the probability of each instance to which instance it should be in and then we return each instance with 4 probabilities and this way we used Gaussian Mixture Model as a feature extraction method

now we pass the new features to our models and see how it react 

In [ ]:
LR_model = LogisticRegression()
LR_model.fit(xtrain_gmm, ytrain_gmm)
y_pred= LR_model.predict(xval_gmm)
lr_acc_gmm  = accuracy_score(yval_gmm,y_pred)
print('validation accuracy : {}'.format(lr_acc_gmm))

you can see the validation acc is wayyy higher 

In [ ]:
num_neighs = np.arange(1,10)
train_accuracy = []
val_accuracy = []
best_accuracy_knn_gmm = 0.0
for i, n in enumerate(num_neighs):
    knn = KNeighborsClassifier(n_neighbors=n)
    knn.fit(xtrain_gmm, ytrain_gmm)
    train_accuracy.append(knn.score(xtrain_gmm, ytrain_gmm))
    val_accuracy.append(knn.score(xval_gmm, yval_gmm))
    print("kNN (k={}): train accuracy={:.5f}, validation accuracy={:.5f}"
          .format(n, train_accuracy[i], val_accuracy[i]))
    if best_accuracy_knn_gmm < val_accuracy[i] :
        best_accuracy_knn_gmm = val_accuracy[i]
        best_knn_gmm = knn
        best_num_neighs_gmm = n

plt.plot(num_neighs, train_accuracy, label='Train')
plt.plot(num_neighs, val_accuracy, label='Validation')
plt.ylabel('Accuracy')
plt.xlabel('Number of neighbors')
plt.legend()
plt.show()

print('Best validation accuracy (k={}): {:.5f}'.format(best_num_neighs_gmm, best_accuracy_knn_gmm))

with just 3 k we got the best validation accuarcy which is not suprising after the feature extraction step 

In [ ]:
estimators = [25, 50, 75, 100]  # number of estimators 
max_depths = [20, 30, 40, 50]
splits = [5, 10, 15]
param = {'n_estimators':estimators, 
         'max_depth':max_depths, 
         'min_samples_split':splits}

rf = RandomForestClassifier(random_state=100)
RandomForestCV = GridSearchCV(estimator=rf, param_grid=param, cv=3)  # cv=5 in default
RandomForestCV.fit(xtrain_gmm, ytrain_gmm)
train_accuracy = RandomForestCV.score(xtrain_gmm, ytrain_gmm)
best_accuracy_rf_gmm = RandomForestCV.score(xval_gmm, yval_gmm)

# Print results
print(RandomForestCV.best_params_)
print("RF: train accuracy={:.5f}, validation accuracy={:.5f}"
      .format(train_accuracy, best_accuracy_rf_gmm))

RF did well but you can see difference of the gridsearch result also the validation acc got lower which is indicating that our model is more genralized 

In [ ]:
# SVM (linear)
svm = SVC(kernel='linear', random_state=100, C=1)
svm.fit(xtrain_gmm, ytrain_gmm)
# Train accuracy
train_accuracy = svm.score(xtrain_gmm, ytrain_gmm)
# Validation accuracy
best_accuracy_svml_gmm = svm.score(xval_gmm, yval_gmm)
print("SVM (linear): train accuracy={:.5f}, validation accuracy={:.5f}"
      .format(train_accuracy, best_accuracy_svml_gmm))

In [ ]:
# SVM (rbf)
svm = SVC(kernel='rbf', gamma='auto', random_state=100, C=1)
svm.fit(xtrain_gmm, ytrain_gmm)
# Train accuracy
train_accuracy = svm.score(xtrain_gmm, ytrain_gmm)
# Validation accuracy
best_accuracy_svmr_gmm = svm.score(xval_gmm, yval_gmm)
print("SVM (rbf): train accuracy={:.5f}, validation accuracy={:.5f}"
      .format(train_accuracy, best_accuracy_svmr_gmm))

svm as well gave me the best accuarcy for both training and validation

In [ ]:
acc_raw = [lr_acc, best_accuracy_rf, best_accuracy_svml, best_accuracy_svmr]
acc_gmm = [lr_acc_gmm, best_accuracy_rf_gmm, best_accuracy_svml_gmm, best_accuracy_svmr_gmm]

res = pd.DataFrame({'Raw': acc_raw, 'GMM': acc_gmm}, index=['LR','RF','SVM (linear)','SVM (rbf)'])
print(res)

this is a comparesion between with GMM and not using it and you can see GM gave the highest accuarcy and each model

I** decided to submit with SVM linear and rbf also with RF**

In [ ]:
svm = SVC(kernel='linear', random_state=100, C=1)
svm.fit(train_gmm, y_train)
ypred = svm.predict(test_gmm)
submission = {'Id': np.arange(1, ypred.shape[0]+1),
             'Solution': ypred}
submission = pd.DataFrame(submission)
submission.to_csv('submission_svm_linear.csv', index=False)

In [ ]:
svm = SVC(kernel='rbf', random_state=100, C=1)
svm.fit(train_gmm, y_train)
ypred = svm.predict(test_gmm)
submission = {'Id': np.arange(1, ypred.shape[0]+1),
             'Solution': ypred}
submission = pd.DataFrame(submission)
submission.to_csv('submission_svm_rbf.csv', index=False)

In [ ]:
rf = RandomForestClassifier(max_depth=20, min_samples_split=5, n_estimators=25)
rf.fit(train_gmm, y_train)
ypred = rf.predict(test_gmm)
submission = {'Id': np.arange(1, ypred.shape[0]+1),
             'Solution': ypred}
submission = pd.DataFrame(submission)
submission.to_csv('submission_rf.csv', index=False)